In [ ]:
from geo_data import data_handler, helpers, svg_handler
import re

In [ ]:
countries = data_handler.load("country", "ne", resolution=110)

regional_groups = data_handler.get_regional_groups()
country_transl = data_handler.get_country_translations()

In [ ]:
def get_country_codes(key):
    country_names = reg_group[key]
    country_codes = []
    for c in country_names:
        country_code = country_transl[country_transl["german"] == c]["code"].iloc[0]
        country_codes.append(country_code)
    assert len(country_names) == len(country_codes)
    return country_codes


save_path = helpers.get_top_directory() / "results" / "regional_groups"

# get countries belonging to a regional group
for reg_group_name, reg_group in regional_groups.items():
    reg_group_id = reg_group_name.lower()
    umlaut_map = str.maketrans({"ä": "ae", "ö": "oe", "ü": "ue","ß": "ss"})
    reg_group_id = reg_group_id.translate(umlaut_map)
    reg_group_id = re.sub(r"[^a-zA-Z0-9._-]+", "_", reg_group_id)

    # get regional group
    reg_country_codes = get_country_codes("core")
    reg_mask = countries["adm0_a3_de"].isin(reg_country_codes)
    reg_countries = countries[reg_mask]
    # get optional countries
    opt_country_codes = get_country_codes("optional")
    opt_mask = countries["adm0_a3_de"].isin(opt_country_codes)
    opt_countries = countries[opt_mask]
    # get other countries
    other_countries = countries[~(reg_mask | opt_mask)]

    center = reg_countries.union_all().centroid
    canvas = svg_handler.OrthoMapSVG(width=500, center=(center.x, center.y))
    canvas.add_sea()
    # other countries
    land_kwargs = canvas.get_kwargs("land")
    canvas.add_gdf(other_countries, "countries", **land_kwargs)
    # highlighted countries
    highlight_color = svg_handler.COLORS["highlight"]
    land_color = land_kwargs["fill"]
    land_kwargs["fill"] = highlight_color
    canvas.add_gdf(reg_countries, reg_group_id, **land_kwargs)
    # optional striped countries
    canvas.add_def(
        svg_handler.DiagonalStripedPattern(
            id="optional-country-pattern", color=(land_color, highlight_color)
        )
    )
    land_kwargs["fill"] = "url(#optional-country-pattern)"
    canvas.add_gdf(opt_countries, f"{reg_group_id}_optional", **land_kwargs)

    canvas.add_shadow()
    canvas.save(save_path / f"{reg_group_id}.svg")